# Format Eval metrics

## Set Up

In [ ]:
import sys
import warnings

sys.path.append("../")
from src.config import BASE_PATH, DEVICE
import pandas as pd
import json
import numpy as np
from shutil import rmtree

print(f"Using device: {DEVICE}")
print(f"Path: {BASE_PATH}")

In [ ]:
class_report_dir = BASE_PATH / "results" / "class_report"
bin_report_dir = BASE_PATH / "results" / "bin_report"

Class report

In [ ]:
for outcome_path in class_report_dir.iterdir():
    outcome_name = outcome_path.stem
    for split_name in ["train", "val", "test"]:
        export_path = (
            BASE_PATH
            / "results"
            / "class_report_tables"
            / outcome_name
            / f"{split_name}.tsv"
        )
        if export_path.exists():
            export_path.unlink()
        export_path.parent.mkdir(exist_ok=True, parents=True)
        cur_rows = []
        with open(outcome_path, "r") as f:
            result_dict = json.load(f)
        sub_dict = result_dict[split_name]
        for model_name, model_result_dict in sub_dict.items():
            new_row = {"model": model_name, **model_result_dict}
            cur_rows.append(new_row)
        cur_df = pd.DataFrame(cur_rows)
        cur_df.to_csv(export_path, sep="\t")

Bin report

In [ ]:
import pandas as pd
import json

for outcome_path in bin_report_dir.iterdir():
    outcome_name = outcome_path.stem
    with open(outcome_path) as f:
        bin_data = json.load(f)
    export_path = BASE_PATH / "results" / "bin_report_tables" / f"{outcome_name}.tsv"
    if export_path.exists():
        export_path.unlink()
    export_path.parent.mkdir(exist_ok=True, parents=True)
    rows = []
    for model, bin_dict in bin_data.items():
        for bin_name, metrics in bin_dict.items():
            rows.append(
                {
                    "Model": model,
                    "Bin": bin_name,
                    "Total N allocated (% all patients)": f"{metrics["n_perc"]["n"]} ({metrics["n_perc"]["perc"]:.1%})",
                    "Positive N allocated (% all positives)": f"{metrics["perc_all_pos"]["n"]} ({metrics["perc_all_pos"]["perc"]:.1%})",
                    "Event Rate": f"{metrics['event_rate_w_CIs']['event_rate']:.1%} ({metrics['event_rate_w_CIs']['lower_CI']:.1%}, {metrics['event_rate_w_CIs']['upper_CI']:.1%})",
                    "Lift": f"{metrics['lift']:.2f}",
                    "Mean Prediction": f"{metrics['mean_model_output']:.1%}",
                    "Thresholds": metrics["thresholds"],
                }
            )

    df = pd.DataFrame(rows)
    df = df.set_index(["Model", "Bin"])
    df.to_csv(export_path, sep="\t")